# Prompts, parámetros y conversación

- Escribir **buenos prompts** (las instrucciones que le das).
- Controlar su **creatividad** con la *temperatura*.
- Limitar la longitud con `max_tokens` (y por qué importa el **coste**).
- Construir un **mini-chatbot con memoria** que recuerda lo que hablasteis.
- Pedir respuestas en **formato JSON**.



## Configuración 

Ejecuta estas dos celdas para instalar la librería, meter tu clave y crear el cliente.

In [2]:
from openai import OpenAI
from getpass import getpass

API_KEY = getpass("Pega aquí tu clave API: ")

PROVEEDOR = "groq"   # "groq" o "openrouter"

if PROVEEDOR == "groq":
    BASE_URL = "https://api.groq.com/openai/v1"
    MODELO   = "llama-3.3-70b-versatile"
else:
    BASE_URL = "https://openrouter.ai/api/v1"
    MODELO   = "meta-llama/llama-3.3-70b-instruct:free"

cliente = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("Cliente listo")

Cliente listo


## Una función para no repetir código

Hasta ahora escribíamos toda la llamada cada vez. Para no repetirnos, creamos una **función** llamada `preguntar` que hace el trabajo pesado. Así, pedir algo al modelo será tan fácil como escribir `preguntar("...")`.

In [3]:
def preguntar(mensaje_usuario, instrucciones_sistema="Eres un asistente util.", temperatura=0.7, max_tokens=500):
    
    respuesta = cliente.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": instrucciones_sistema},
            {"role": "user",   "content": mensaje_usuario},
        ],
        temperature=temperatura,
        max_tokens=max_tokens,
    )
    return respuesta.choices[0].message.content

print(preguntar("Dame un consejo para dormir mejor."))

**Consejo para dormir mejor: Establece una rutina de sueño**

Una de las formas más efectivas de mejorar la calidad del sueño es establecer una rutina de sueño consistente. Esto significa ir a la cama y levantarse a la misma hora todos los días, incluyendo los fines de semana.

Aquí hay algunos pasos que puedes seguir:

1. **Establece un horario de sueño**: Decide a qué hora quieres acostarte y levantarte, y trata de ceñirte a ese horario todos los días.
2. **Crea un ambiente relajante**: Asegúrate de que tu habitación esté oscura, fresca y silenciosa. Puedes usar cortinas, un ventilador o un dispositivo de ruido blanco para crear un ambiente relajante.
3. **Evita la estimulación antes de dormir**: Trata de evitar actividades estimulantes como ver televisión, usar el teléfono o realizar ejercicio intenso antes de acostarte.
4. **Relájate antes de dormir**: Puedes probar técnicas de relajación como la meditación, la respiración profunda o la lectura de un libro para calmar tu mente y cu

## La anatomía de un buen prompt

Un **prompt** es el texto que le das al modelo. La diferencia entre un prompt flojo y uno bueno es **enorme**. Un buen prompt suele tener 4 ingredientes:

| Ingrediente | Pregunta que responde | Ejemplo |
|-------------|-----------------------|---------|
| **Rol** | ¿Quién eres? | "Eres un nutricionista." |
| **Tarea** | ¿Qué quiero? | "Crea un menú semanal." |
| **Formato** | ¿Cómo lo quiero? | "En una tabla, con desayuno, comida y cena." |
| **Ejemplos / contexto** | ¿Algún detalle? | "Vegetariano, sin gluten." |

Veamos la diferencia entre pedir mal y pedir bien. Ejecuta las dos celdas y compara.

In [4]:
print("--- PROMPT VAGO ---")
print(preguntar("Háblame de comida."))

--- PROMPT VAGO ---
**La Comida: Un Mundo de Sabores y Culturas**

La comida es una parte fundamental de nuestra vida diaria. No solo nos proporciona la energía necesaria para funcionar, sino que también es una forma de expresar nuestra cultura, tradición y creatividad. En este sentido, la comida es un lenguaje universal que puede unir a personas de diferentes orígenes y trasfondos.

**Tipos de Comida**

Existen una variedad de tipos de comida que se pueden clasificar de acuerdo a su origen, ingredientes y preparación. Algunos de los tipos más comunes son:

* **Comida internacional**: incluye platos y técnicas de cocina de diferentes países y culturas, como la comida china, italiana, mexicana, india, etc.
* **Comida tradicional**: se refiere a los platos y recetas que han sido pasados de generación en generación y son característicos de una región o país específico.
* **Comida saludable**: se enfoca en la preparación de platos que son ricos en nutrientes y bajos en grasas y azúcares, c

In [5]:
prompt_bueno = """Eres un nutricionista.
Crea un menú de UN día (desayuno, comida y cena).
Preséntalo como una lista con viñetas.
Debe ser vegetariano y fácil de preparar."""

print("--- PROMPT BUENO ---")
print(preguntar(prompt_bueno))

--- PROMPT BUENO ---
¡Claro! Aquí te presento un menú vegetariano y fácil de preparar para un día:

* **Desayuno**: Avena con frutas y nueces - una taza de avena cocida con leche vegetal, mezclada con frutas frescas (plátano, fresa, arándano) y espolvoreada con nueces picadas (almendras o nueces de macadamia)
* **Comida**: Ensalada de quinoa con verduras asadas - una taza de quinoa cocida, mezclada con verduras asadas (brócoli, zanahoria, pimiento rojo) y aderezada con un aliño de limón y aceite de oliva
* **Cena**: Tacos de verduras con guacamole - tortillas de maíz rellenas con verduras salteadas (calabacín, cebolla, tomate), acompañadas de un guacamole fresco hecho con aguacate, limón y cilantro

Espero que disfrutes de este menú vegetariano y fácil de preparar. ¡Buen provecho!


## La temperatura.

La **temperatura** controla cuánto "se arriesga" el modelo al elegir palabras:

- **Temperatura baja (0.0 – 0.3):** respuestas **predecibles y centradas**. Ideal para datos, clasificar, resúmenes, código.
- **Temperatura alta (0.8 – 1.2):** respuestas **creativas y variadas**. Ideal para lluvia de ideas, historias, nombres.

Vamos a pedir lo mismo con dos temperaturas distintas y a ver cómo cambia.

In [6]:
peticion = "Inventa un nombre original para una cafetería."

print("TEMPERATURA BAJA (0.1) — dos intentos:")
print(" 1.", preguntar(peticion, temperatura=0.1))
print(" 2.", preguntar(peticion, temperatura=0.1))

print("TEMPERATURA ALTA (1.1) — dos intentos:")
print(" 1.", preguntar(peticion, temperatura=1.1))
print(" 2.", preguntar(peticion, temperatura=1.1))

TEMPERATURA BAJA (0.1) — dos intentos:
 1. ¡Claro! Un nombre original para una cafetería podría ser:

**"Café con Alma"**

Este nombre sugiere que la cafetería no solo ofrece bebidas deliciosas, sino que también es un lugar donde las personas pueden encontrar un ambiente acogedor y cálido, como si estuvieran en casa. La palabra "alma" implica que la cafetería tiene un toque personal y emotivo, lo que puede atraer a clientes que buscan una experiencia más profunda que solo un café rápido.

Otras opciones podrían ser:

* **"El Rincón del Sabor"**
* **"La Casa del Café"**
* **"Brew & Co."**
* **"El Jardín de las Delicias"**
* **"Café con Historia"**

¿Te gustaría que te sugiera más nombres?
 2. ¡Claro! Un nombre original para una cafetería podría ser:

**"Café con Alma"**

Este nombre sugiere que la cafetería no solo ofrece bebidas deliciosas, sino que también tiene un toque de personalidad y carácter. La palabra "alma" implica que la cafetería tiene un espíritu cálido y acogedor, lo que 

Fíjate: con temperatura **baja**, los dos intentos se parecen mucho. Con temperatura **alta**, son más distintos y atrevidos.

## `max_tokens` y el coste.

`max_tokens` limita **cuánto** puede responder el modelo (en tokens de salida). Sirve para dos cosas:
- **Evitar respuestas eternas** cuando solo quieres algo corto.
- **Controlar el coste**: en los servicios de pago, cada token de salida cuesta dinero.

Vamos a comprobar cuántos tokens gasta una respuesta. Esta vez llamamos al modelo "a mano" para poder leer el `usage`.

In [7]:
respuesta = cliente.chat.completions.create(
    model=MODELO,
    messages=[{"role": "user", "content": "Explica qué es internet en 2 frases."}],
    max_tokens=80,
)
print(respuesta.choices[0].message.content)
print("\n— Tokens usados:", respuesta.usage.total_tokens,
      "(entrada:", respuesta.usage.prompt_tokens,
      "+ salida:", respuesta.usage.completion_tokens, ")")

Internet es una red global de redes de comunicación que conecta millones de ordenadores y dispositivos en todo el mundo, permitiendo el intercambio de información y la comunicación entre usuarios a través de una variedad de protocolos y tecnologías. A través de internet, las personas pueden acceder a una amplia gama de servicios y recursos, como sitios web, correo electrónico

— Tokens usados: 126 (entrada: 46 + salida: 80 )


## Memoria: un mini-chatbot

Hasta ahora, cada llamada era **independiente**: el modelo **no recuerda** lo anterior. Si le dices tu nombre y luego se lo preguntas, no lo sabrá.

¿Cómo le damos memoria? Muy sencillo: **le reenviamos toda la conversación cada vez**. Guardamos los mensajes en una lista que va creciendo. Esto es **clave** para entender cómo funcionan los chatbots y los agentes.

Primero, veamos el problema (sin memoria):

In [9]:
print(preguntar("Me llamo Diego y me encanta la F1."))
print("---")
print(preguntar("¿Cómo me llamo y qué me gusta?"))

¡Hola Diego! Me alegra conocerte. La Fórmula 1 es un deporte emocionante y lleno de acción. ¿Cuál es tu equipo favorito en la F1? ¿Tienes un piloto preferido? ¿Has tenido la oportunidad de asistir a algún Gran Premio en vivo? Estoy aquí para charlar contigo sobre la F1 y responder a cualquier pregunta que tengas. ¡Vamos a hablar de carreras!
---
Lo siento, pero no tengo información sobre ti. Soy un asistente diseñado para proporcionar información y responder a preguntas en general, pero no tengo acceso a información personal sobre los usuarios. Si deseas compartir algo sobre ti, estaré encantado de charlar contigo y conocer más sobre tus intereses y preferencias. ¿Hay algo en particular que te gustaría hablar o preguntar?


Como ves, no se acuerda. Ahora lo arreglamos **acumulando los mensajes** en una lista llamada `conversacion`.

In [10]:
conversacion = [
    {"role": "system", "content": "Eres un asistente cercano y breve."}
]

def chatear(texto_usuario):
    conversacion.append({"role": "user", "content": texto_usuario})
    
    respuesta = cliente.chat.completions.create(model=MODELO, messages=conversacion)
    contenido = respuesta.choices[0].message.content
    
    conversacion.append({"role": "assistant", "content": contenido})
    return contenido

print(chatear("Me llamo Diego y me encanta la F1."))
print("---")
print(chatear("¿Cómo me llamo y qué me gusta?"))

¡Hola Diego! Me alegra conocerte. La F1 es un deporte emocionante. ¿Tienes un equipo o piloto favorito en la Fórmula 1?
---
Te llamas Diego y te gusta la Fórmula 1 (F1).


El "secreto" de cualquier chatbot es justo esto: **mantener la lista de mensajes y reenviarla**.

Prueba de mas llamadas.

In [11]:
print(chatear("Recomiéndame un libro relacionado con lo que me gusta."))

Un libro relacionado con la Fórmula 1 que te podría gustar es "Senna" de Christopher Hilton. Sin embargo, si buscas algo más general sobre la F1, "La Fórmula 1: Historia y Técnica" es un libro excelente que cubre la historia y los detalles técnicos del deporte.

También te recomiendo "Mi vida en la F1" de Fernando Alonso, donde el piloto español comparte sus experiencias y perspectivas sobre el mundo de la Fórmula 1.

¿Te gustaría que te recomiende más libros?


## Pedir respuestas en JSON (preparándonos para los agentes)

A veces no queremos un texto bonito, sino **datos ordenados** que el programa pueda usar. El formato estándar para eso es **JSON**: parejas de "campo: valor".

Para conseguirlo, **se lo pedimos claramente en el prompt**. Esto será fundamental en el Notebook 3, porque así el agente nos dirá *qué herramienta quiere usar*.

In [13]:
import json

prompt_json = """Extrae los datos de esta frase y devuélvelos SOLO como JSON,
sin ningún texto adicional ni explicaciones.
Campos: nombre, lugar de nacimiento, ciudad.

Frase: "Hola, soy Diego, nací en Lugo y vivo en Vitoria-Gasteiz."
"""

respuesta_texto = preguntar(prompt_json, instrucciones_sistema="Devuelves únicamente JSON válido.", temperatura=0)
print("Texto recibido del modelo:")
print(respuesta_texto)

Texto recibido del modelo:
{"nombre": "Diego", "lugar de nacimiento": "Lugo", "ciudad": "Vitoria-Gasteiz"}


Ahora la parte importante: una función **defensiva** para convertir ese texto en datos reales de Python, aunque el modelo haya añadido algo de más.

In [15]:
respuesta_texto

'{"nombre": "Diego", "lugar de nacimiento": "Lugo", "ciudad": "Vitoria-Gasteiz"}'

In [18]:
respuesta_texto.rfind("}")

78

In [21]:
def extraer_json(texto):
    
    if texto is None:
        return None
    
    inicio = texto.find("{")
    fin = texto.rfind("}")
    if inicio == -1 or fin == -1:
        return None
    fragmento = texto[inicio:fin + 1]
    try:
        return json.loads(fragmento)
    except json.JSONDecodeError:
        return None

print(type(respuesta_texto))
datos = extraer_json(respuesta_texto)
print("Convertido a datos de Python:", datos)
print(type(datos))

print("El nombre es:", datos.get("nombre"))
print("La ciudad es:", datos.get("ciudad"))


<class 'str'>
Convertido a datos de Python: {'nombre': 'Diego', 'lugar de nacimiento': 'Lugo', 'ciudad': 'Vitoria-Gasteiz'}
<class 'dict'>
El nombre es: Diego
La ciudad es: Vitoria-Gasteiz
